# Model Confidence Set (MCS)

The **Model Confidence Set** (Hansen, Lunde & Nason, 2011) determines the smallest set of models that contains the best forecasting model with a given level of confidence $(1 - \alpha)$.

Unlike pairwise tests (DM), the MCS provides a **multiple comparison** procedure that controls the familywise error rate. Think of it as a confidence interval, but for *models* instead of parameters.

This notebook covers:
1. The MCS concept and intuition
2. MCS with the **Range** statistic ($T_R$)
3. MCS with the **Semi-Quadratic** statistic ($T_{SQ}$)
4. Interpreting elimination sequences and p-values
5. Sensitivity to the confidence level $\alpha$

**Reference:**
- Hansen, P.R., Lunde, A. & Nason, J.M. (2011). "The Model Confidence Set." *Econometrica*, 79(2), 453-497.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "..", ".."))

from forecastbox.evaluation import model_confidence_set, MCSResult
from forecastbox.evaluation import diebold_mariano
from utils.helpers import load_inflation_forecasts, load_m4_sample

## 1. MCS Concept

The MCS procedure works by **sequential elimination**:

1. Start with the full set of $K$ models $\mathcal{M}_0 = \{1, 2, \ldots, K\}$
2. Test $H_0$: all models in the current set have equal predictive ability
3. If $H_0$ is rejected, eliminate the worst-performing model
4. Repeat until $H_0$ is not rejected

The surviving models form the **Model Confidence Set** — with probability $\geq 1 - \alpha$, this set contains the truly best model.

**Analogy:** Just as a 95% confidence interval contains the true parameter value with 95% probability, a 90% MCS contains the true best model with 90% probability.

In [ ]:
# Load data and prepare forecasts dictionary
df = load_inflation_forecasts()
actual = df["actual"].values

forecasts = {
    "ARIMA": df["fc_arima"].values,
    "ETS": df["fc_ets"].values,
    "VAR": df["fc_var"].values,
    "Naive": df["fc_naive"].values,
    "Drift": df["fc_drift"].values,
}

print(f"Number of models: {len(forecasts)}")
print(f"Sample size: {len(actual)}")

# Compute MSE for each model
print("\nModel MSE Rankings:")
print("-" * 30)
mse_scores = {}
for name, fc in forecasts.items():
    mse = np.mean((actual - fc) ** 2)
    mse_scores[name] = mse
    print(f"  {name:<10}: {mse:.6f}")

best_model = min(mse_scores, key=mse_scores.get)
print(f"\nBest model by MSE: {best_model}")

## 2. MCS with Range Statistic

The **Range statistic** $T_R$ is based on the maximum absolute t-statistic across all pairwise loss differentials:

$$T_R = \max_{i,j \in \mathcal{M}} |t_{ij}|$$

where $t_{ij}$ tests whether models $i$ and $j$ have equal loss. This statistic is sensitive to the single worst pair of models in the set.

In [ ]:
# MCS with Range statistic at alpha=0.10
mcs_range = model_confidence_set(
    actual, forecasts, alpha=0.10, loss="mse",
    statistic="range", n_boot=5000, seed=42,
)

print(mcs_range.summary())
print(f"\nElimination order: {mcs_range.elimination_order}")
print(f"Included models: {mcs_range.included_models}")

## 3. MCS with Semi-Quadratic Statistic

The **Semi-Quadratic statistic** $T_{SQ}$ aggregates evidence across all models:

$$T_{SQ} = \sum_{i \in \mathcal{M}} \left( \frac{1}{K} \sum_{j \in \mathcal{M}} \bar{d}_{ij} \right)^2 / \hat{V}_i$$

This statistic has more power when multiple models are jointly inferior, while $T_R$ is more powerful against a single outlier model.

In [ ]:
# MCS with Semi-Quadratic statistic
mcs_sq = model_confidence_set(
    actual, forecasts, alpha=0.10, loss="mse",
    statistic="semi_quadratic", n_boot=5000, seed=42,
)

print(mcs_sq.summary())

# Compare the two statistics
print("\n\nComparison: Range vs Semi-Quadratic")
print("=" * 50)
print(f"{'Model':<12} {'p (Range)':>12} {'p (SQ)':>12}")
print("-" * 36)
all_models = list(forecasts.keys())
for m in all_models:
    pr = mcs_range.pvalues.get(m, float("nan"))
    ps = mcs_sq.pvalues.get(m, float("nan"))
    print(f"{m:<12} {pr:>12.4f} {ps:>12.4f}")

print(f"\nRange MCS:  {mcs_range.included_models}")
print(f"SQ MCS:     {mcs_sq.included_models}")

## 4. Interpreting MCS Results

The MCS p-value for each model represents the **largest significance level at which that model would still be included** in the MCS. Models with high p-values are more likely to be among the best.

The **elimination order** shows which models were removed first (worst) to last.

In [ ]:
# Visualize elimination sequence and p-values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MCS p-values as bar chart (sorted)
sorted_pvals = sorted(mcs_range.pvalues.items(), key=lambda x: x[1], reverse=True)
names_sorted = [x[0] for x in sorted_pvals]
pvals_sorted = [x[1] for x in sorted_pvals]
colors = ["green" if n in mcs_range.included_models else "salmon" for n in names_sorted]

axes[0].barh(range(len(names_sorted)), pvals_sorted, color=colors, edgecolor="black")
axes[0].set_yticks(range(len(names_sorted)))
axes[0].set_yticklabels(names_sorted)
axes[0].axvline(0.10, color="red", linestyle="--", label="alpha=0.10")
axes[0].set_xlabel("MCS p-value")
axes[0].set_title("MCS p-values (Range statistic)")
axes[0].legend()

# Right: Elimination sequence
elim_order = mcs_range.elimination_order
elim_pvals = [mcs_range.pvalues[m] for m in elim_order]
surviving = [m for m in all_models if m in mcs_range.included_models]

step_labels = [f"Step {i+1}: drop {m}" for i, m in enumerate(elim_order)]
step_labels.append(f"Final: keep {surviving}")

axes[1].plot(range(1, len(elim_pvals) + 1), elim_pvals, "ro-", markersize=8)
axes[1].axhline(0.10, color="blue", linestyle="--", alpha=0.7, label="alpha=0.10")
axes[1].set_xticks(range(1, len(elim_pvals) + 1))
axes[1].set_xticklabels([m for m in elim_order], rotation=45)
axes[1].set_xlabel("Eliminated Model")
axes[1].set_ylabel("p-value at elimination")
axes[1].set_title("Sequential Elimination")
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary table
print("Elimination Sequence:")
print("-" * 50)
for i, model in enumerate(mcs_range.elimination_order):
    print(f"  Step {i+1}: Eliminate '{model}' (p-value = {mcs_range.pvalues[model]:.4f})")
print(f"  Final MCS: {mcs_range.included_models}")

## 5. MCS at Different Confidence Levels

The choice of $\alpha$ controls the trade-off between **selectivity** and **coverage**:
- **Smaller $\alpha$** (e.g., 0.05): Larger MCS, more conservative — harder to exclude models
- **Larger $\alpha$** (e.g., 0.25): Smaller MCS, more aggressive — easier to exclude inferior models

Let's see how the MCS changes across $\alpha \in \{0.05, 0.10, 0.25\}$:

In [ ]:
# MCS at different confidence levels
alphas = [0.05, 0.10, 0.25]

print("MCS at Different Confidence Levels (Range statistic)")
print("=" * 60)

for alpha in alphas:
    mcs_result = model_confidence_set(
        actual, forecasts, alpha=alpha, loss="mse",
        statistic="range", n_boot=5000, seed=42,
    )
    included = mcs_result.included_models
    excluded = mcs_result.excluded_models
    print(f"\nalpha = {alpha:.2f} ({(1 - alpha):.0%} confidence):")
    print(f"  Included ({len(included)}): {included}")
    print(f"  Excluded ({len(excluded)}): {excluded}")

# Visualization: which models are in/out at each alpha
fig, ax = plt.subplots(figsize=(10, 4))

# For each model, show its MCS p-value vs thresholds
pvals = mcs_range.pvalues
sorted_models = sorted(pvals.keys(), key=lambda m: pvals[m], reverse=True)

y_pos = range(len(sorted_models))
bars = ax.barh(y_pos, [pvals[m] for m in sorted_models], color="steelblue", edgecolor="black")
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_models)

for alpha, color, ls in zip(alphas, ["red", "orange", "green"], ["--", "-.", ":"]):
    ax.axvline(alpha, color=color, linestyle=ls, linewidth=2, label=f"alpha={alpha}")

ax.set_xlabel("MCS p-value")
ax.set_title("Model p-values vs Significance Thresholds")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("\nInterpretation: A model is INCLUDED in the MCS if its p-value >= alpha.")
print("Lower alpha => more models survive => more conservative test.")

## Exercise 1: Apply MCS to M4 series models

Load the `m4_sample.csv` dataset. For each of the 6 M4 series, run MCS on the 3 forecast models. How often does each model appear in the MCS?

In [ ]:
# Exercise 1 - MCS for each M4 series
m4 = load_m4_sample()
series_ids = sorted(m4["series_id"].unique())
m4_model_names = ["Model1", "Model2", "Model3"]

print("MCS for Each M4 Series (alpha=0.10, Range statistic)")
print("=" * 65)

# Track which models are included in MCS for each series
inclusion_table: dict[str, list[str]] = {sid: [] for sid in series_ids}
pvalue_table: list[dict[str, object]] = []

for sid in series_ids:
    subset = m4[m4["series_id"] == sid]
    actual_s = subset["actual"].values
    fc_dict = {
        "Model1": subset["fc_model1"].values,
        "Model2": subset["fc_model2"].values,
        "Model3": subset["fc_model3"].values,
    }

    mcs_result = model_confidence_set(
        actual_s, fc_dict, alpha=0.10, loss="mse",
        statistic="range", n_boot=5000, seed=42,
    )

    inclusion_table[sid] = mcs_result.included_models

    row: dict[str, object] = {"Series": sid}
    for m in m4_model_names:
        row[f"p({m})"] = mcs_result.pvalues.get(m, float("nan"))
    row["MCS"] = ", ".join(mcs_result.included_models)
    row["Eliminated"] = ", ".join(mcs_result.elimination_order)
    pvalue_table.append(row)

    print(f"\n{sid} (T={len(actual_s)}):")
    # MSE for reference
    for mname in m4_model_names:
        mse = np.mean((actual_s - fc_dict[mname]) ** 2)
        in_mcs = "IN MCS" if mname in mcs_result.included_models else "excluded"
        print(f"  {mname}: MSE={mse:.6f}, p={mcs_result.pvalues.get(mname, float('nan')):.4f} [{in_mcs}]")
    print(f"  Elimination order: {mcs_result.elimination_order}")

# Summary table
print("\n\nSummary Table: MCS p-values and Inclusion")
print("=" * 75)
summary_df = pd.DataFrame(pvalue_table)
print(summary_df.to_string(index=False, float_format="%.4f"))

# Model inclusion frequency
print("\n\nModel Inclusion Frequency in MCS")
print("-" * 40)
for model in m4_model_names:
    count = sum(1 for sid in series_ids if model in inclusion_table[sid])
    pct = count / len(series_ids) * 100
    print(f"  {model}: included in {count}/{len(series_ids)} series ({pct:.0f}%)")

print("\nInterpretation: Models that appear in the MCS across most series are")
print("robustly competitive. A model excluded from the MCS for a given series")
print("is statistically significantly worse than the best model for that series.")

## Exercise 2: Compare MCS results with pairwise DM tests

Run both MCS and all pairwise DM tests on the inflation forecasts. Do the DM pairwise results agree with the MCS elimination order? When might they disagree?

In [ ]:
# Exercise 2 - Compare MCS with DM pairwise
model_list = list(forecasts.keys())
n_models = len(model_list)

# 1. Pairwise DM tests
print("Pairwise DM Tests (two-sided, MSE loss, alpha=0.05)")
print("=" * 65)

dm_pval_matrix = np.ones((n_models, n_models))
dm_stat_matrix = np.zeros((n_models, n_models))

for i in range(n_models):
    for j in range(n_models):
        if i != j:
            res = diebold_mariano(actual, forecasts[model_list[i]],
                                  forecasts[model_list[j]], h=1, loss="mse")
            dm_pval_matrix[i, j] = res.pvalue
            dm_stat_matrix[i, j] = res.statistic

dm_df = pd.DataFrame(dm_pval_matrix, index=model_list, columns=model_list)
print(dm_df.round(4).to_string())

# DM-based ranking: count significant losses per model
print("\nDM-based Ranking (number of models significantly BETTER than each):")
dm_ranking = []
for i, m in enumerate(model_list):
    n_losses = sum(
        1 for j in range(n_models)
        if i != j and dm_pval_matrix[i, j] < 0.05 and dm_stat_matrix[i, j] > 0
    )
    n_wins = sum(
        1 for j in range(n_models)
        if i != j and dm_pval_matrix[i, j] < 0.05 and dm_stat_matrix[i, j] < 0
    )
    dm_ranking.append({"Model": m, "Sig. Wins": n_wins, "Sig. Losses": n_losses,
                       "Net": n_wins - n_losses})

dm_rank_df = pd.DataFrame(dm_ranking).sort_values("Net", ascending=False)
print(dm_rank_df.to_string(index=False))

# 2. MCS results (already computed above)
print(f"\nMCS Elimination Order: {mcs_range.elimination_order}")
print(f"MCS Included:          {mcs_range.included_models}")

# 3. Comparison
print("\n\nComparison: DM Ranking vs MCS")
print("=" * 65)
print(f"{'Model':<12} {'DM Net Wins':>12} {'MCS p-value':>12} {'In MCS?':>10}")
print("-" * 48)
for _, row in dm_rank_df.iterrows():
    m = row["Model"]
    pval = mcs_range.pvalues.get(m, float("nan"))
    in_mcs = "Yes" if m in mcs_range.included_models else "No"
    print(f"{m:<12} {row['Net']:>12} {pval:>12.4f} {in_mcs:>10}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: DM p-value heatmap
im = axes[0].imshow(dm_pval_matrix, cmap="RdYlGn", vmin=0, vmax=1)
axes[0].set_xticks(range(n_models))
axes[0].set_yticks(range(n_models))
axes[0].set_xticklabels(model_list, rotation=45)
axes[0].set_yticklabels(model_list)
for i in range(n_models):
    for j in range(n_models):
        color = "white" if dm_pval_matrix[i, j] < 0.3 else "black"
        axes[0].text(j, i, f"{dm_pval_matrix[i, j]:.2f}", ha="center", va="center",
                     color=color, fontsize=9)
plt.colorbar(im, ax=axes[0], label="p-value")
axes[0].set_title("DM Pairwise p-values")

# Right: MCS p-values
sorted_models = sorted(mcs_range.pvalues.keys(),
                       key=lambda m: mcs_range.pvalues[m], reverse=True)
colors = ["green" if m in mcs_range.included_models else "salmon" for m in sorted_models]
axes[1].barh(range(len(sorted_models)),
             [mcs_range.pvalues[m] for m in sorted_models],
             color=colors, edgecolor="black")
axes[1].set_yticks(range(len(sorted_models)))
axes[1].set_yticklabels(sorted_models)
axes[1].axvline(0.10, color="red", linestyle="--", label="alpha=0.10")
axes[1].set_xlabel("MCS p-value")
axes[1].set_title("MCS Model p-values")
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nDiscussion:")
print("- DM tests are pairwise: each comparison is independent with no multiplicity")
print("  correction. With K models, we make K(K-1)/2 comparisons, inflating the")
print("  familywise error rate (multiple testing problem).")
print("- MCS controls the familywise error rate via bootstrap, so its conclusions")
print("  are jointly valid. This makes MCS more conservative than raw DM.")
print("- A model may 'win' many DM pairwise tests but still be in the MCS with")
print("  other models, because MCS accounts for the uncertainty in ALL comparisons.")
print("- Discrepancies arise when DM rejects for a pair but MCS keeps both models:")
print("  this reflects the multiplicity adjustment — individual significance does")
print("  not guarantee joint significance.")
print("- Use DM for focused pairwise questions; use MCS when selecting among many.")